# Ejercicio Tarea

![Logo Itq](img/ITQ.png)

**Nombre:** Angelica Paredes

**Fecha:** 12/11/2026

**Link del repositorio:** <a href="https://github.com/AngelicaPar/FundamentosMachineLearning26-27">Link Repositorio</a>

# Normalización y estandarización de datos médicos con CRISP DM

## Planteamiento del problema
Un equipo hospitalario desea explorar cómo se distribuyen las características de las hospitalizaciones de pacientes con diabetes y cómo se relacionan descriptivamente con la readmisión. Antes de construir modelos, necesita conocer la calidad de los datos y disponer de versiones comparables de sus variables cuantitativas.
El estudiante descargará el conjunto completo, documentará sus características, justificará la limpieza y producirá dos versiones independientes de las mismas variables: una normalizada y otra estandarizada. No se requiere entrenar ni evaluar modelos.
Alcance metodológico: fases 1 y 2 de CRISP-DM completas para esta práctica; fase 3 hasta selección, limpieza y escalamiento. La normalización y la estandarización pertenecen a la fase 3.


# Paso 1. Instalación del paquete de descarga

In [10]:
%pip install -q ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [11]:
from ucimlrepo import fetch_ucirepo
 
# Descargar el conjunto de datos desde UCI.
dataset = fetch_ucirepo(id=296)
 
# Obtener una copia de todos los datos.
df = dataset.data.original.copy()
 
# Verificar que cumple con el tamaño solicitado.
print("Filas y columnas:", df.shape)
assert len(df) > 100_000, "El dataset debe superar los 100 000 registros."
 
# Visualizar los primeros registros.
display(df.head())
 
# Consultar el diccionario de variables.
display(dataset.variables)
 
# Guardar una copia local para desarrollar la actividad.
df.to_csv("hospitalizaciones_diabetes_original.csv", index=False)

C:\Users\USUARIO\anaconda3\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Filas y columnas: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


,name,role,type,demographic,description,units,missing_values
0,encounter_id,ID,,None,Unique identifier of an encounter,None,no
1,patient_nbr,ID,,None,Unique identifier of a patient,None,no
2,race,Feature,Categorical,Race,"Values: Caucasian, Asian, African American, Hi...",None,yes
3,gender,Feature,Categorical,Gender,"Values: male, female, and unknown/invalid",None,no
4,age,Feature,Categorical,Age,"Grouped in 10-year intervals: [0, 10), [10, 20...",None,no
5,weight,Feature,Categorical,None,Weight in pounds.,None,yes
6,admission_type_id,Feature,Categorical,None,Integer identifier corresponding to 9 distinct...,None,no
7,discharge_disposition_id,Feature,Categorical,None,Integer identifier corresponding to 29 distinc...,None,no
8,admission_source_id,Feature,Categorical,None,Integer identifier corresponding to 21 distinc...,None,no
9,time_in_hospital,Feature,Integer,None,Integer number of days between admission and d...,None,no


In [12]:
# Registro de los datos descargados dimensiones
# y versión de las bibliotecas utilizadas.
import sys, datetime
import pandas, numpy, matplotlib, seaborn, sklearn

print("Fecha de descarga:", datetime.date.today())
print("Dimensiones:", df.shape)
print("Python:", sys.version.split()[0])
for lib in (pandas, numpy, matplotlib, seaborn, sklearn):
    print(lib.__name__, lib.__version__)


Fecha de descarga: 2026-11-12
Dimensiones: (101766, 50)
Python: 3.13.9
pandas 2.3.3
numpy 2.3.5
matplotlib 3.10.6
seaborn 0.13.2
sklearn 1.7.2


# Paso 2. Fase 1 Comprensión del negocio

##Describir el problema 
##Objetivo general
##Dos preguntas de analisis
##Explique los limites del periodo 1999-2008

# Paso 3 Fase 2 Comprensión de los datos

In [13]:
import numpy as np
import pandas as pd

print("Hospitalizaciones (filas):", len(df))
print("encounter_id únicos:", df['encounter_id'].nunique())
print("Pacientes únicos (patient_nbr):", df['patient_nbr'].nunique())
df.info()

Hospitalizaciones (filas): 101766
encounter_id únicos: 101766
Pacientes únicos (patient_nbr): 71518
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      99493 non-null   object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    3197 non-null    object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                61510 non-null   object
 11  medical_specialty         51817 non-null   object
 12  num_lab_proced

In [14]:
nulos = df.isna().sum()
interrog = (df == '?').sum()

faltantes = pd.DataFrame({
    'nulos': nulos,
    'signos_?': interrog,
})
faltantes['total_faltantes'] = faltantes['nulos'] + faltantes['signos_?']
faltantes['porcentaje'] = (faltantes['total_faltantes'] / len(df) * 100).round(2)

display(faltantes[faltantes['total_faltantes'] > 0]
        .sort_values('porcentaje', ascending=False))

,nulos,signos_?,total_faltantes,porcentaje
weight,98569,0,98569,96.86
max_glu_serum,96420,0,96420,94.75
A1Cresult,84748,0,84748,83.28
medical_specialty,49949,0,49949,49.08
payer_code,40256,0,40256,39.56
race,2273,0,2273,2.23
diag_3,1423,0,1423,1.40
diag_2,358,0,358,0.35
diag_1,21,0,21,0.02


In [8]:
for col in ['max_glu_serum', 'A1Cresult']:
    print(col)
    print(df[col].value_counts(dropna=False), '\n')

max_glu_serum
max_glu_serum
NaN     96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64 

A1Cresult
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64 

